### Verificar base y predictores candidatos

In [1]:
from pathlib import Path
import pandas as pd

project_root= Path.cwd()
if not (project_root / "data" / "processed").exists():
    project_root = project_root.parent
data_path= (project_root/ "data"/ "processed"/ "demanda_base_analitica.csv")
model_df= pd.read_csv(data_path,parse_dates=["utc_timestamp", "target_timestamp"])
print("Dimensiones:", model_df.shape)
print("\nColumnas:")
print(model_df.columns.tolist())
print("\nDistribución de periodos:")
print(model_df["period"].value_counts(dropna=False))
print("\nPrimeras observaciones:")
display(model_df.head())

Dimensiones: (131472, 25)

Columnas:
['utc_timestamp', 'country', 'demand_mw', 'target_timestamp', 'target_demand_mw', 'period', 'local_timestamp', 'target_local_timestamp', 'target_hour', 'target_day', 'target_is_weekend', 'target_month', 'lag_48h', 'lag_72h', 'lag_168h', 'rolling_mean_24h', 'rolling_mean_168h', 'has_observed_target', 'has_complete_lags', 'has_complete_rolling_24h', 'has_complete_rolling_168h', 'has_complete_candidate_features', 'last_available_timestamp', 'target_outside_dataset', 'time_diff']

Distribución de periodos:
period
development    105120
holdout         26280
NaN                72
Name: count, dtype: int64

Primeras observaciones:


,utc_timestamp,country,demand_mw,target_timestamp,target_demand_mw,period,local_timestamp,target_local_timestamp,target_hour,target_day,...,rolling_mean_24h,rolling_mean_168h,has_observed_target,has_complete_lags,has_complete_rolling_24h,has_complete_rolling_168h,has_complete_candidate_features,last_available_timestamp,target_outside_dataset,time_diff
0,2015-01-01 00:00:00+00:00,DE,41151.0,2015-01-02 00:00:00+00:00,40577.0,development,2015-01-01T01:00:00+01:00,2015-01-02T01:00:00+01:00,1,4,...,NaN,NaN,True,False,False,False,False,2019-12-31 23:00:00+00:00,False,NaN
1,2015-01-01 01:00:00+00:00,DE,40135.0,2015-01-02 01:00:00+00:00,39707.0,development,2015-01-01T02:00:00+01:00,2015-01-02T02:00:00+01:00,2,4,...,NaN,NaN,True,False,False,False,False,2019-12-31 23:00:00+00:00,False,0 days 01:00:00
2,2015-01-01 02:00:00+00:00,DE,39106.0,2015-01-02 02:00:00+00:00,39504.0,development,2015-01-01T03:00:00+01:00,2015-01-02T03:00:00+01:00,3,4,...,NaN,NaN,True,False,False,False,False,2019-12-31 23:00:00+00:00,False,0 days 01:00:00
3,2015-01-01 03:00:00+00:00,DE,38765.0,2015-01-02 03:00:00+00:00,40391.0,development,2015-01-01T04:00:00+01:00,2015-01-02T04:00:00+01:00,4,4,...,NaN,NaN,True,False,False,False,False,2019-12-31 23:00:00+00:00,False,0 days 01:00:00
4,2015-01-01 04:00:00+00:00,DE,38941.0,2015-01-02 04:00:00+00:00,42382.0,development,2015-01-01T05:00:00+01:00,2015-01-02T05:00:00+01:00,5,4,...,NaN,NaN,True,False,False,False,False,2019-12-31 23:00:00+00:00,False,0 days 01:00:00


In [2]:
candidate_predictors= ["country","demand_mw","lag_48h","lag_72h","lag_168h","rolling_mean_24h","rolling_mean_168h","target_hour","target_day","target_month"]
target= "target_demand_mw"
required_columns= candidate_predictors + [target]

# Verificar que todas las variables existen
missing_columns = [
    col for col in required_columns
    if col not in model_df.columns
]
assert not missing_columns, f"Columnas faltantes: {missing_columns}"

# Revisar tipo de variable y valores faltantes
predictor_audit= pd.DataFrame({"dtype": model_df[candidate_predictors].dtypes.astype(str),"missing": model_df[candidate_predictors].isna().sum(),"missing_pct": (model_df[candidate_predictors].isna().mean() * 100).round(2)})
display(predictor_audit)
print("Variable objetivo:", target)
print("Predictores candidatos:", len(candidate_predictors))

,dtype,missing,missing_pct
country,str,0,0.00
demand_mw,float64,48,0.04
lag_48h,float64,120,0.09
lag_72h,float64,192,0.15
lag_168h,float64,480,0.37
rolling_mean_24h,float64,280,0.21
rolling_mean_168h,float64,1582,1.20
target_hour,int64,0,0.00
target_day,int64,0,0.00
target_month,int64,0,0.00


Variable objetivo: target_demand_mw
Predictores candidatos: 10


### Métricas y criterios de comparación

In [3]:
development_df= model_df[model_df["period"] == "development"].copy()
country_scales= (development_df.groupby("country")["target_demand_mw"].mean().rename("mean_demand_mw").to_frame())
country_scales["n_valid"]= (development_df.groupby("country")["target_demand_mw"].count())
display(country_scales)

,mean_demand_mw,n_valid
country,,
DE,55833.754652,35040
ES,28701.707107,35023
FR,54068.605309,35016


La métrica principal será NMAE macro

$$NMAE_{macro}=\frac{1}{3}\sum_{c\in\{DE,ES,FR\}}\frac{MAE_{c}}{\overline{Y}_{c,train}}$$

Donde el denominador es la demanda promedio observada en el conjunto de entrenamiento de cada ventana temporal. Un valor menor está asociado a un menor desempeño.

Algunas métricas complementarias pueden ser: MAE y RMSE por país en MW.

Los criterios de comparación que se usarán son:

- Los modelos se comparan sobre los mismos valores observados válidos.
- Se informa el número de observaciones evaluadas por país.
- Los normalizadores se calularán con los datos del entrenamiento unicamente.
- La selección se basará en el NMAE macro 

### Diseño de validación temporal

In [4]:
# Definición de las ventanas temporales
validation_windows = [
    {"name": "W1","validation_start": "2016-01-01","validation_end": "2017-01-01"},
    {"name": "W2","validation_start": "2017-01-01","validation_end": "2018-01-01"},
    {"name": "W3","validation_start": "2018-01-01","validation_end": "2019-01-01"}
]
validation_partitions = {}
partition_summary = []
for window in validation_windows:
    start= pd.Timestamp(window["validation_start"], tz="UTC")
    end= pd.Timestamp(window["validation_end"], tz="UTC")
    train_mask= ((model_df["period"] == "development")& (model_df["utc_timestamp"] < start)& (model_df["target_timestamp"] < start)& model_df["target_demand_mw"].notna())
    validation_mask= ((model_df["period"] == "development")& (model_df["utc_timestamp"] >= start)& (model_df["utc_timestamp"] < end))
    train_df= model_df.loc[train_mask].copy()
    validation_df= model_df.loc[validation_mask].copy()
    assert not train_df.empty
    assert not validation_df.empty
    assert (train_df["target_timestamp"].max()< validation_df["utc_timestamp"].min())
    validation_partitions[window["name"]] = {"train": train_df,"validation": validation_df}

    for country in ["DE", "ES", "FR"]:
        train_country= train_df[train_df["country"] == country]
        validation_country= validation_df[validation_df["country"] == country]
        partition_summary.append({"window": window["name"],"country": country,"n_train": len(train_country),"n_validation": len(validation_country),"n_validation_target_valid": (validation_country["target_demand_mw"].notna().sum()),"last_train_target": train_df["target_timestamp"].max(),"first_validation_origin": (validation_df["utc_timestamp"].min())})

partition_summary_df= pd.DataFrame(partition_summary)
display(partition_summary_df)

,window,country,n_train,n_validation,n_validation_target_valid,last_train_target,first_validation_origin
0,W1,DE,8736,8784,8784,2015-12-31 23:00:00+00:00,2016-01-01 00:00:00+00:00
1,W1,ES,8719,8784,8784,2015-12-31 23:00:00+00:00,2016-01-01 00:00:00+00:00
2,W1,FR,8736,8784,8784,2015-12-31 23:00:00+00:00,2016-01-01 00:00:00+00:00
3,W2,DE,17520,8760,8760,2016-12-31 23:00:00+00:00,2017-01-01 00:00:00+00:00
4,W2,ES,17503,8760,8760,2016-12-31 23:00:00+00:00,2017-01-01 00:00:00+00:00
5,W2,FR,17520,8760,8747,2016-12-31 23:00:00+00:00,2017-01-01 00:00:00+00:00
6,W3,DE,26280,8736,8736,2017-12-31 23:00:00+00:00,2018-01-01 00:00:00+00:00
7,W3,ES,26263,8736,8736,2017-12-31 23:00:00+00:00,2018-01-01 00:00:00+00:00
8,W3,FR,26267,8736,8725,2017-12-31 23:00:00+00:00,2018-01-01 00:00:00+00:00


Aquí tenemos que:

- W1: entrenamiento 2015, validación 2016.
- W2: entrenamiento 2015-2016, validación 2017.
- W3: entrenamiento 2015-2017, validación 2018.

Cada país tiene observaciones de entrenamiento y validación. Las diferencias en `n_validation_target_valid` corresponden a respuestas faltantes y no necesariamente errores en la partición